# BigQuery & Apache Iceberg 심층 성능 및 파일 레이아웃 분석 노트북

본 노트북은 구축된 **50GB 대용량 4대 테이블 (`native_weblog`, `managed_iceberg_weblog`, `external_iceberg_weblog`, `metastore_iceberg_weblog`)**을 대상으로 쿼리 실증 분석, 메타데이터/스토리지 파싱 및 용어 체계를 다룹니다.

---

## 📖 1. 핵심 개념 및 용어 정의 (Glossary)

### 1.1 Partition Pruning vs Predicate Pushdown 차이점
- **Partition Pruning (파티션 프루닝)**:
  - **대상**: 테이블의 **파티셔닝 컬럼 (`event_date`)**
  - **원리**: 쿼리의 `WHERE` 절 조건(`WHERE event_date BETWEEN '2026-07-03' AND '2026-07-05'`) 구동 시, 엔진이 조건에 해당하지 않는 파티션 디렉터리(`event_date=2026-07-01/` 등) 전체를 쿼리 스캔 대상에서 물리적으로 제외(Skip)합니다.
- **Predicate Pushdown (프레디킷 푸시다운)**:
  - **대상**: 파티셔닝 컬럼이 아닌 **일반 필터 컬럼 또는 클러스터링 컬럼 (`user_id`, `event_type`)**
  - **원리**: 쿼리 조건절(`WHERE user_id = 'USER_10500' AND event_type = 'PURCHASE'`)을 스토리지 메타데이터 레이어(Capacitor Zone Maps / Parquet Column Min-Max Stats)에 하향 전달하여, 조건에 부합하지 않는 데이터 블록 / Parquet Data File / Row Group 전체의 I/O를 통째로 패스(Skip)하는 기법입니다.

### 1.2 Native Table 및 Managed Iceberg가 Predicate Pushdown에서 성능 우위를 가지는 이유
- **클러스터링(Clustering/Sorting) 레이아웃의 유무**:
  - `native_weblog`와 `managed_iceberg_weblog`는 DDL 상에 **`CLUSTER BY user_id, event_type`** 이 적용되어 스토리지 블록 내에 데이터가 물리적으로 정렬/그룹화되어 있습니다. 따라서 Predicate Pushdown 구동 시 min/max bounds 범위가 매우 조밀하여 99.9% 이상의 블록을 즉시 스킵합니다.
  - 반면 External/Metastore Iceberg는 데이터 적재 시 클러스터 정렬(Sort Order)이 미적용된 채 분산 수집된 경우, Parquet 파일별 Min/Max 범위가 넓어 스킵 효율이 상대적으로 낮아집니다. (외부 Iceberg에 `write.sort.order = user_id, event_type` 적용 시 스킵률 대폭 상승)

### 1.3 Iceberg/Parquet 파일 레이아웃 용어 정의

- **BigQuery Native Capacitor vs Iceberg Parquet 읽기 차이**:
  - **Capacitor**: BigQuery 독자적인 인메모리/디스크 칼럼너 포맷. Dictionary/RLE 인코딩이 BQ Execution Slot 버퍼에 Direct Vectorization 매핑되어 CPU 디코딩 오버헤드가 0에 수렴합니다.
  - **Parquet**: 오픈소스 칼럼너 포맷. BQ Reader가 GCS 스토리지에서 Parquet Footer를 파싱하고 Snappy/ZSTD 압축 해제 및 Row Group 디코딩을 진행하므로 CPU 및 메모리 소모가 수반됩니다.
- **Parquet File Size & Small Files (소형 파일 문제)**:
  - 파일 크기가 너무 작은 경우(< 32MB) 수천~수십만 개의 소형 파일이 누적되면 Iceberg Manifest 파일 비대화 및 GCS HTTP GET 요청 횟수 폭증으로 쿼리 Latency가 저하됩니다. (권장 파일 크기: **128MB ~ 512MB**, 기본 권장: **256MB**)
- **Row Group Size & Row-Group Skipping**:
  - Parquet 파일 내부의 레코드 집합 단위(권장: **64MB ~ 128MB**). 각 Row Group Footer에는 컬럼별 Min/Max 통계가 저장되어 있어 쿼리 조건에 부합하지 않는 Row Group 전체를 통째로 패스하는 **Row-Group Level Skipping**이 가능합니다.
- **Manifest / Metadata 개수 및 Compaction 기준**:
  - **Manifest File**: Apache Iceberg에서 데이터 파일들의 경로, 파티션 정보, 컬럼별 min/max 통계를 담고 있는 Avro 포맷 메타데이터 파일입니다.
  - **Compaction (압축/병합)**: 소형 데이터 파일 수량이 **1,000개 이상 누적**되거나, Merge-on-Read (MoR) Delete File 비율이 **15%를 초과**할 때 Spark `rewrite_data_files` / `rewrite_manifests` 명령을 통해 파일 크기를 최적화하는 보수작업입니다.

In [ ]:
# =========================================================================
# ⚙️ [프로젝트 설정 및 4대 대상 테이블 실시간 행 수 정밀 검증]
# =========================================================================
import os
import time
import uuid
import google.auth
import pandas as pd
import numpy as np
import pyarrow as pa
from google.cloud import bigquery
from google.cloud import storage
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType, TimestampType, DateType, DoubleType

# 💡 외부 사용자는 아래 USER_PROJECT_ID 변수에 본인의 GCP Project ID를 지정해 주세요.
# 예: USER_PROJECT_ID = "your-gcp-project-id" (None 지정 시 gcloud 로그인 ADC 환경 자동 탐지)
USER_PROJECT_ID = None

PROJECT_ID = USER_PROJECT_ID or os.getenv("GCP_PROJECT_ID")

try:
    credentials, auth_project = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
    if not PROJECT_ID and auth_project:
        PROJECT_ID = auth_project
except Exception:
    credentials = None

if not PROJECT_ID:
    PROJECT_ID = "your-gcp-project-id"

DATASET_NAME = "bq_iceberg_benchmark_ds"
CONNECTION_NAME = "lakehouse-iceberg-conn"
REGION = os.getenv("GCP_REGION", "asia-northeast3")

# 💡 고정 정적 버킷 이름 (Static Bucket Name - 난수화 제거)
BUCKET_NAME = f"{PROJECT_ID}-bq-iceberg-demo-bucket"

temp_bq = bigquery.Client(project=PROJECT_ID, credentials=credentials)
try:
    ds_obj = temp_bq.get_dataset(f"{PROJECT_ID}.{DATASET_NAME}")
    REGION = ds_obj.location
except Exception:
    pass

gcs_client = storage.Client(project=PROJECT_ID, credentials=credentials)

# 4대 검증 대상 테이블 타겟 Dictionary
table_targets = {
    "Native (Capacitor)": f"{PROJECT_ID}.{DATASET_NAME}.native_weblog",
    "BQ Managed Iceberg": f"{PROJECT_ID}.{DATASET_NAME}.managed_iceberg_weblog",
    "Lakehouse External Iceberg": f"{PROJECT_ID}.{DATASET_NAME}.external_iceberg_weblog",
    "BQ Metastore Iceberg": f"{PROJECT_ID}.{DATASET_NAME}.metastore_iceberg_weblog"
}

print("=========================================================")
print(f"🎯 설정된 GCP Project ID : {PROJECT_ID}")
print(f"🎯 설정된 GCP Region     : {REGION}")
print(f"🎯 고정 GCS 버킷 이름    : {BUCKET_NAME}")
print(f"🎯 벤치마킹 Dataset ID   : {DATASET_NAME}")
print("=========================================================\n")

bq_client = bigquery.Client(project=PROJECT_ID, location=REGION, credentials=credentials)

# =========================================================================
# 📊 [4대 대상 테이블 실시간 행 수(SELECT COUNT(*)) 정밀 측정 및 동기화 검증]
# =========================================================================
print("📊 4대 대상 테이블 실시간 행 수(SELECT COUNT(*)) 검증 중...\n")

table_inspection_results = []

for t_name, t_id in table_targets.items():
    try:
        # 실시간 SQL SELECT COUNT(*) 측정 (Iceberg/External REST API num_rows 0 반환 문제 해결)
        count_sql = f"SELECT COUNT(*) as total_rows FROM `{t_id}`"
        query_job = bq_client.query(count_sql)
        row_res = list(query_job.result())
        actual_rows = row_res[0]["total_rows"] if row_res else 0
        
        tbl_obj = bq_client.get_table(t_id)
        table_type = tbl_obj.table_type
        
        table_inspection_results.append({
            "테이블 명칭": t_name,
            "Table ID": t_id,
            "테이블 타입": table_type,
            "실시간 검증 행 수 (COUNT(*))": f"{actual_rows:,} 개",
            "1.5억 건 동기화 상태": "✅ Pass (1.5억 건 완료)" if actual_rows >= 150000000 else f"⚠️ ({actual_rows:,}건)"
        })
        print(f"  ✅ [{t_name}] -> 실시간 SQL 검증 행 수: {actual_rows:,} 개 (테이블 타입: {table_type})")
    except Exception as e:
        print(f"  ℹ️ [{t_name}] 테이블 미생성 또는 수집 안내: {e}")

df_inspection = pd.DataFrame(table_inspection_results)
if not df_inspection.empty:
    print("\n=========================================================")
    print("📋 4대 테이블 행 수 검증 종합 결과 (1.5억 건 기준)")
    print("=========================================================")
df_inspection


## 🧪 [3단계: 질문 1 & 2 검증] Iceberg 타입별 성능 및 비용 정밀 수집

본 섹션에서는 50GB 4대 테이블을 대상으로 다음 3가지 쿼리 패턴을 구동하여 **`bytes_processed_mb`**, **`slot_millis`**, **`elapsed_time_sec`**, **`query_cost_usd`** 수치와 `job.query_plan` 세부 Stage 및 Execution 프로필을 정밀 측정합니다.

- **시나리오 A (Partition Pruning)**: `WHERE event_date BETWEEN '2026-07-03' AND '2026-07-05'`
- **시나리오 B (Predicate Pushdown & Column Pruning)**: `WHERE user_id = 'USER_10500' AND event_type = 'PURCHASE'`
- **시나리오 C (Full Scan Aggregation)**: `GROUP BY device_os, event_type` 전체 집계 (스캔 성능 및 `slot_ms` 비유)

In [ ]:
# =========================================================================
# [3단계: 4대 테이블 대상 정밀 쿼리 벤치마크 및 job.query_plan 프로파일링]
# =========================================================================
COST_PER_TB_USD = 6.25  # BigQuery On-Demand Pricing ($6.25 per TB)

def execute_profiled_query(scenario_title, table_type_name, sql_template, target_table_id):
    job_config = bigquery.QueryJobConfig(use_query_cache=False) # 캐시 완전 비활성화
    formatted_sql = sql_template.format(table_id=target_table_id)
    
    start_t = time.time()
    query_job = bq_client.query(formatted_sql, job_config=job_config)
    results = list(query_job.result())
    elapsed_sec = time.time() - start_t
    
    bytes_processed = query_job.total_bytes_processed or 0
    slot_millis = query_job.slot_millis or 0
    bytes_mb = bytes_processed / (1024 * 1024)
    cost_usd = (bytes_processed / (1024**4)) * COST_PER_TB_USD
    
    # query_plan 요약 추출
    plan_stages = len(query_job.query_plan) if query_job.query_plan else 0
    
    return {
        "Scenario": scenario_title,
        "Table Type": table_type_name,
        "Elapsed (sec)": round(elapsed_sec, 3),
        "Bytes Processed (MB)": round(bytes_mb, 2),
        "Slot Milliseconds (ms)": slot_millis,
        "Est. Cost ($ USD)": round(cost_usd, 6),
        "Plan Stages": plan_stages
    }

metrics_data = []

# 쿼리 시나리오 정의
sql_scenarios = [
    ("Scenario A: Partition Pruning", """
        SELECT event_date, COUNT(*) as cnt, SUM(amount) as total_amt
        FROM `{table_id}`
        WHERE event_date BETWEEN '2026-07-03' AND '2026-07-05'
        GROUP BY event_date
    """),
    ("Scenario B: Predicate Pushdown", """
        SELECT user_id, event_type, amount
        FROM `{table_id}`
        WHERE user_id = 'USER_10500' AND event_type = 'PURCHASE'
    """),
    ("Scenario C: Full Scan Aggregation", """
        SELECT device_os, event_type, COUNT(*) as cnt, AVG(amount) as avg_amt
        FROM `{table_id}`
        GROUP BY device_os, event_type
    """)
]

print("🚀 50GB 4대 테이블 벤치마킹 쿼리 프로파일링 구동 중...\n")

for sc_title, sc_sql in sql_scenarios:
    for t_name, t_id in table_targets.items():
        try:
            m = execute_profiled_query(sc_title, t_name, sc_sql, t_id)
            metrics_data.append(m)
            print(f"  [{sc_title} | {t_name}] -> {m['Elapsed (sec)']}s, {m['Bytes Processed (MB)']} MB, {m['Slot Milliseconds (ms)']} ms")
        except Exception as e:
            print(f"  ❌ [{sc_title} | {t_name}] 쿼리 실패: {e}")

df_results = pd.DataFrame(metrics_data)
print("\n✅ 벤치마킹 데이터 수집 완수!")
df_results


In [ ]:
# =========================================================================
# [3-1단계: 4대 테이블 벤치마킹 시각화 (Bytes, Elapsed Time, Slot Millis, Cost)]
# =========================================================================
import matplotlib.pyplot as plt
import seaborn as sns

df_plot = None

if 'df_results' in globals() and isinstance(df_results, pd.DataFrame) and not df_results.empty:
    df_plot = df_results.rename(columns={
        "Scenario": "scenario", "Table Type": "table_type",
        "Bytes Processed (MB)": "bytes_processed_mb", "Elapsed (sec)": "elapsed_time_sec",
        "Slot Milliseconds (ms)": "slot_millis", "Est. Cost ($ USD)": "query_cost_usd"
    })
elif 'metrics_data' in globals() and metrics_data:
    df_plot = pd.DataFrame(metrics_data).rename(columns={
        "Scenario": "scenario", "Table Type": "table_type",
        "Bytes Processed (MB)": "bytes_processed_mb", "Elapsed (sec)": "elapsed_time_sec",
        "Slot Milliseconds (ms)": "slot_millis", "Est. Cost ($ USD)": "query_cost_usd"
    })
elif 'metrics_list' in globals() and metrics_list:
    df_plot = pd.DataFrame(metrics_list)
elif 'benchmark_results' in globals() and benchmark_results:
    df_plot = pd.DataFrame(benchmark_results)

# 4대 테이블 전체 데이터 보장
if df_plot is None or df_plot.empty or len(df_plot['table_type'].unique()) < 4:
    print("ℹ️ 4대 테이블 시각화 데이터를 구성하기 위해 쿼리 프로파일링을 자동 실행합니다...")
    COST_PER_TB_USD = 6.25
    table_targets = {
        "Native (Capacitor)": f"{PROJECT_ID}.{DATASET_NAME}.native_weblog",
        "BQ Managed Iceberg": f"{PROJECT_ID}.{DATASET_NAME}.managed_iceberg_weblog",
        "Lakehouse External Iceberg": f"{PROJECT_ID}.{DATASET_NAME}.external_iceberg_weblog",
        "BQ Metastore Iceberg": f"{PROJECT_ID}.{DATASET_NAME}.metastore_iceberg_weblog"
    }
    sql_scenarios = [
        ("Scenario A: Partition Pruning", "SELECT event_date, COUNT(*) as cnt, SUM(amount) as total_amt FROM `{table_id}` WHERE event_date BETWEEN '2026-07-03' AND '2026-07-05' GROUP BY event_date"),
        ("Scenario B: Predicate Pushdown", "SELECT user_id, event_type, amount FROM `{table_id}` WHERE user_id = 'USER_10500' AND event_type = 'PURCHASE'"),
        ("Scenario C: Full Scan Aggregation", "SELECT device_os, event_type, COUNT(*) as cnt, AVG(amount) as avg_amt FROM `{table_id}` GROUP BY device_os, event_type")
    ]
    auto_metrics = []
    for sc_title, sc_sql in sql_scenarios:
        for t_name, t_id in table_targets.items():
            try:
                job_config = bigquery.QueryJobConfig(use_query_cache=False)
                st = time.time()
                q_job = bq_client.query(sc_sql.format(table_id=t_id), job_config=job_config)
                list(q_job.result())
                el = time.time() - st
                bp = q_job.total_bytes_processed or 0
                sm = q_job.slot_millis or 0
                mb = bp / (1024 * 1024)
                co = (bp / (1024**4)) * COST_PER_TB_USD
                auto_metrics.append({
                    "scenario": sc_title, "table_type": t_name,
                    "elapsed_time_sec": round(el, 3), "bytes_processed_mb": round(mb, 2),
                    "slot_millis": sm, "query_cost_usd": round(co, 6)
                })
            except Exception as e:
                print(f"❌ 쿼리 실행 불가 [{sc_title} - {t_name}]: {e}")
    df_plot = pd.DataFrame(auto_metrics)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(4, 1, figsize=(14, 24))

# 1. Bytes Processed (MB)
sns.barplot(
    data=df_plot,
    x="bytes_processed_mb",
    y="scenario",
    hue="table_type",
    ax=axes[0],
    palette="viridis"
)
axes[0].set_title("1. Bytes Processed (Data Scan Size in MB)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Processed MB", fontsize=12)
axes[0].legend(title="Table Type", bbox_to_anchor=(1.05, 1), loc='upper left')

# 2. Elapsed Time (Seconds)
sns.barplot(
    data=df_plot,
    x="elapsed_time_sec",
    y="scenario",
    hue="table_type",
    ax=axes[1],
    palette="rocket"
)
axes[1].set_title("2. Elapsed Wall-Clock Time (Execution Time in Seconds)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Elapsed Time (sec)", fontsize=12)
axes[1].legend(title="Table Type", bbox_to_anchor=(1.05, 1), loc='upper left')

# 3. Slot Millis (ms)
sns.barplot(
    data=df_plot,
    x="slot_millis",
    y="scenario",
    hue="table_type",
    ax=axes[2],
    palette="magma"
)
axes[2].set_title("3. Slot Millis (CPU Compute Consumption in ms)", fontsize=14, fontweight="bold")
axes[2].set_xlabel("Slot Millis (ms)", fontsize=12)
axes[2].legend(title="Table Type", bbox_to_anchor=(1.05, 1), loc='upper left')

# 4. Estimated Query Cost (USD)
sns.barplot(
    data=df_plot,
    x="query_cost_usd",
    y="scenario",
    hue="table_type",
    ax=axes[3],
    palette="crest"
)
axes[3].set_title("4. Estimated Query Cost ($ USD @ $6.25/TB On-Demand)", fontsize=14, fontweight="bold")
axes[3].set_xlabel("Estimated Cost ($ USD)", fontsize=12)
axes[3].legend(title="Table Type", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig("deepdive_analysis_visualization.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ bq_iceberg_analysis 3-1단계 시각화 4종 그래프 및 deepdive_analysis_visualization.png 가 정상 생성되었습니다.")


## 🔬 [4단계: 질문 3 검증] Iceberg/Parquet 파일 레이아웃 & GCS 스토리지 분석

본 섹션에서는 GCS 스토리지 상에 빌드된 Apache Iceberg 파일 레이아웃(`external_iceberg_warehouse/` 및 `metastore_iceberg_warehouse/`)을 직접 파싱하여 다음 세부 항목을 실증적으로 확인합니다.

1. **Parquet File Size & Small Files 문제 검증**:
   - GCS 상의 Parquet 데이터 파일들의 평균 크기 및 분포 파싱
   - 소형 파일(<32MB) 수량 및 manifest 파일 개수 파싱
2. **PyArrow를 활용한 Parquet Footer & Row Group Size 분석**:
   - `pyarrow.parquet.ParquetFile` 메타데이터 파싱으로 Row Group 개수, Row Group 단위의 레코드 수, Row Group 크기 측정
   - Row Group Min/Max Zone Maps를 통한 **Row-Group Level Skipping** 및 **Column Pruning** 가능 범위 검증
3. **Compaction 가이드라인**:
   - 소형 파일 수량 및 MoR Delete 파일 비율에 따른 Compaction 실행 조건 정립

In [ ]:
# =========================================================================
# [4단계: GCS 상의 Iceberg/Parquet 파일 레이아웃 정밀 파싱]
# =========================================================================
import pyarrow.parquet as pq

BUCKET_NAME = f"{PROJECT_ID}-bq-iceberg-demo-pyiceberg"
bucket = gcs_client.get_bucket(BUCKET_NAME)

print(f"📦 [{BUCKET_NAME}] GCS 버킷 상의 Iceberg 파일 구조 탐색 중...\n")

blobs = list(bucket.list_blobs(prefix="external_iceberg_warehouse/"))

metadata_files = [b for b in blobs if b.name.endswith(".metadata.json")]
manifest_files = [b for b in blobs if b.name.endswith(".avro")]
parquet_files = [b for b in blobs if b.name.endswith(".parquet")]

parquet_sizes_mb = [b.size / (1024 * 1024) for b in parquet_files]
small_files_cnt = sum(1 for s in parquet_sizes_mb if s < 32.0)

print(f"📊 [Iceberg 메타데이터 & 데이터파일 통계]")
print(f"  - Metadata JSON 파일 개수 : {len(metadata_files):,} 개")
print(f"  - Manifest Avro 파일 개수 : {len(manifest_files):,} 개")
print(f"  - Parquet Data 파일 개수  : {len(parquet_files):,} 개")
if parquet_sizes_mb:
    print(f"  - 평균 Parquet 파일 크기 : {np.mean(parquet_sizes_mb):.2f} MB")
    print(f"  - 최소/최대 Parquet 크기 : {np.min(parquet_sizes_mb):.2f} MB ~ {np.max(parquet_sizes_mb):.2f} MB")
    print(f"  - Small Files (<32MB) 수 : {small_files_cnt:,} 개 ({small_files_cnt/len(parquet_files)*100:.1f}%)")

# 첫 번째 Parquet 파일의 Footer 상세 파싱 (PyArrow)
if parquet_files:
    sample_blob = parquet_files[0]
    local_sample = "/tmp/sample_iceberg.parquet"
    sample_blob.download_to_filename(local_sample)
    
    pq_file = pq.ParquetFile(local_sample)
    print(f"\n🔍 [Sample Parquet File Footer 파싱 ({sample_blob.name})]")
    print(f"  - Num Row Groups : {pq_file.num_row_groups}")
    print(f"  - Total Row Count: {pq_file.metadata.num_rows:,} 행")
    
    # Row Group 0 세부 정보
    rg0 = pq_file.metadata.row_group(0)
    print(f"  - Row Group 0 레코드 수: {rg0.num_rows:,} 행")
    print(f"  - Row Group 0 Total Size: {rg0.total_byte_size / (1024*1024):.2f} MB")
    print(f"  - Column Pruning 지원 수: {rg0.num_columns} 컬럼 통계 파싱 가능")


## 📌 5. 심층 결론 및 아키텍처/파일 레이아웃 가이드라인

---

### 💡 질문 1 결론: BigQuery Metastore vs Lakehouse External vs BigQuery Managed Iceberg

1. **BigQuery Managed Iceberg**:
   - **권장 상황**: BigQuery 중심의 데이터 레이크하우스를 구축하면서 오픈 표준 포맷인 Apache Iceberg 호환성을 유지해야 하는 프로젝트.
   - **이점**: BigQuery 모놀리식 메타데이터 카탈로그 엔진이 트랜잭션과 메타데이터 커밋을 전담하므로 Native Capacitor 테이블과 동일한 레벨의 초고속 쿼리 성능과 완전 자동화된 DML/MERGE 및 Compaction 혜택을 누립니다.
2. **Lakehouse External Iceberg**:
   - **권장 상황**: 이미 Apache Spark, Flink, Trino 파이프라인에서 GCS 버킷 상에 Apache Iceberg 테이블을 생성 및 운영 중이며, BigQuery에서는 Ad-hoc Read-only BI 쿼리만 구동할 때.
   - **이점**: 별도의 ETL이나 데이터 복제 없이 GCS 상의 `metadata.json` URIs를 직접 지정하여 즉각적인 쿼리가 가능합니다.
3. **BigQuery Metastore Iceberg**:
   - **권장 상황**: Dataproc Metastore 또는 Hive Metastore 기반의 중앙 카탈로그 서비스를 보유하고 수많은 이종 데이터엔진(Spark, Presto, Flink, BigQuery)이 실시간으로 메타데이터를 공유해야 할 때.
   - **이점**: Central Metastore 카탈로그 레이어를 표준화하여 거버넌스를 강화합니다.

---

### 💡 질문 2 결론: 타입별 성능 및 비용 차이

| 성능 축 | Native Table (Capacitor) | BQ Managed Iceberg | Lakehouse External Iceberg | BQ Metastore Iceberg |
| :--- | :--- | :--- | :--- | :--- |
| **Partition Pruning** | 최상 (Internal BQ Metadata) | 최상 (Manifest List Skip) | 최상 (PartitionSpec `event_date` Skip) | 최상 (PartitionSpec `event_date` Skip) |
| **Predicate Pushdown** | 최상 (Capacitor Zone Maps) | 우수 (Iceberg lower/upper) | 우수 (Iceberg lower/upper) | 우수 (Iceberg lower/upper) |
| **Metadata Pruning** | Internal Zero-latency | 2-tier Internal Cache | GCS HTTP File Read Latency | Metastore API Sync |
| **`slot_ms` 소모량** | **최소** (Direct Vectorization) | 상대적 높음 (Iceberg Metadata & Parquet Parsing) | 상대적 높음 (GCS IO & Parsing) | 우수 |
| **조회 비용 ($ USD)** | On-Demand ($6.25/TB) | On-Demand ($6.25/TB) | On-Demand + GCS Egress | On-Demand |

---

### 💡 질문 3 결론: Parquet/Iceberg 파일 레이아웃 최적화 가이드라인

1. **Native Capacitor vs Iceberg Parquet 읽기 차이**:
   - Native Capacitor는 BigQuery Engine 버퍼에 Direct Vectorization 매핑되어 CPU 디코딩 레이턴시가 0에 수렴하지만, Parquet 포맷은 GCS 스토리지에서의 Parquet Footer 파싱 및 Snappy/ZSTD 압축 해제 오버헤드로 인해 `slot_ms` 소모가 상대적으로 높습니다.
2. **Parquet File Size & Small Files 최적화**:
   - 파일 크기가 32MB 미만으로 작아질 경우 수천 개의 소형 파일로 인해 Manifest 파일이 비대해지고 GCS HTTP GET 요청 폭증으로 Latency가 하락합니다.
   - **권장 파키 파일 크기**: **128MB ~ 512MB** (기본 추천값: **256MB**)
3. **Row Group Size & Row-Group Skipping**:
   - **권장 Row Group 크기**: **64MB ~ 128MB**
   - Row Group Footer에 명시된 Min/Max 컬럼 통계를 기반으로 **Row-Group Level Skipping**이 발동하여 I/O를 절감시킵니다.
4. **Compaction 임계치 가이드라인**:
   - 소형 파일 수량이 **1,000개 이상**이거나, Merge-on-Read (MoR) Delete File 비율이 **15%를 초과**하는 경우 Spark `rewrite_data_files` / `rewrite_manifests` Compaction 명령 구동을 필수 권장합니다.
